# Project 3 — EEG + EMG Fusion

Pairs with [Project 2: EEG motor imagery](01_eeg_motor_imagery.ipynb) in the multimodal-intent-decoding repo. Reuses Project 2's Riemannian/CSP EEG pipeline and adds an EMG branch plus fusion.

**Dataset**: Jeong et al. 2020, *GigaScience* — 60-channel EEG, 7-channel EMG, 4-channel EOG, 25 healthy participants. 11 movement tasks (arm-reaching in 6 directions, hand-grasping of 3 objects, wrist-twisting with 2 motions), 3 sessions/participant one week apart, ~82,500 trials total (~3,300/participant). Both real-movement and motor-imagery conditions recorded, same labels, same clock, both modalities. DOI: 10.1093/gigascience/giaa098.

Backup: WAY-EEG-GAL (32-ch EEG, EMG from 5 arm/hand muscles, 3D hand/object position + contact force, 3,936 grasp-lift trials, 12 participants).

## The framing correction that makes this project good
Naively concatenating features and comparing fusion against EMG-alone finds almost no gain — EMG has far better SNR for movement classification and dominates every fusion model. That's a known result, not a bug, and reporting it naively reads as naive. This notebook is framed as a **robustness study** instead:

1. Train three models on identical splits: **EMG-only, EEG-only, fused**.
2. Fuse at three levels: **feature concatenation**, **decision-level** (probability averaging + stacking), **intermediate** (two-branch network with a learned gating weight).
3. **Systematically degrade the EMG branch** and plot accuracy vs. degradation for all three models — drop channels 7→1, inject Gaussian noise at decreasing SNR, attenuate amplitude to simulate weak residual activation (amputee / fatigued user). The crossover point where fusion overtakes EMG-alone is the clinically meaningful result: the whole reason to add EEG to a prosthetic is that EMG is unreliable in the population that needs it.
4. **Exploit timing asymmetry**: movement-related cortical activity precedes muscle activation by a few hundred ms. Plot accuracy vs. time-relative-to-cue per modality — if EEG carries usable intent before EMG onset, that's an argument for EEG as an early trigger and EMG as confirmation (a real control architecture).
5. **Mandatory control**: EMG contaminates EEG. Show the EEG branch isn't just reading muscle artifact — bandpass EEG 8–30 Hz, ICA against EOG, and an ablation testing the EEG model on pre-movement windows only.


## 1. Setup

In [ ]:
# Colab setup
# !pip install -q mne pyriemann scikit-learn torch numpy pandas matplotlib scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
from scipy.signal import butter, sosfiltfilt, iirnotch, tf2sos

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, f1_score

try:
    from pyriemann.estimation import Covariances
    from pyriemann.tangentspace import TangentSpace
    HAS_PYRIEMANN = True
except ImportError:
    HAS_PYRIEMANN = False

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

mne.set_log_level("WARNING")
RNG_SEED = 0
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 2. Config

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    eeg_fs: int = 2500          # Jeong et al. native EEG sampling rate
    emg_fs: int = 2500          # EMG recorded on the same clock
    eeg_channels: int = 60
    emg_channels: int = 7
    eeg_bp: tuple = (8.0, 30.0)  # motor-imagery band
    emg_bp: tuple = (20.0, 450.0)
    emg_notch: float = 60.0      # Korea-based acquisition, 60 Hz mains — verify against the paper's site
    epoch_pre_s: float = 1.0     # seconds before cue
    epoch_post_s: float = 2.0    # seconds after cue
    n_classes: int = 11

CFG = Config()

# Google Drive mount (Colab). Skip if DATA_DIR is already populated locally.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = "/content/drive/MyDrive/bhuvan research project/jeong_eeg_emg"
except ImportError:
    DATA_DIR = "./data/jeong_eeg_emg"


## 3. Data loading

Jeong et al. distribute per-subject, per-session files (originally GDF/mat via GigaDB). This loader assumes a preprocessed layout of one `.npz` per (subject, session) with `eeg`, `emg`, `eog`, `labels`, `trial_onsets_samples` arrays — adjust the loader to match whatever raw format you download from GigaDB.

In [ ]:
import os, glob

def list_subject_session_files(cfg=CFG, data_dir=None):
    data_dir = data_dir or DATA_DIR
    pattern = os.path.join(data_dir, "sub-*_ses-*.npz")
    files = sorted(glob.glob(pattern))
    if not files:
        print(f"No files under {data_dir} matching sub-*_ses-*.npz — download the Jeong et al. "
              f"GigaScience dataset from GigaDB (doi: 10.1093/gigascience/giaa098), convert to this "
              f"layout, and set DATA_DIR.")
    return files


def load_session(path):
    d = np.load(path, allow_pickle=True)
    return {
        "eeg": d["eeg"].astype(np.float32),          # (T, 60)
        "emg": d["emg"].astype(np.float32),           # (T, 7)
        "eog": d["eog"].astype(np.float32) if "eog" in d else None,  # (T, 4)
        "labels": d["labels"],                        # (n_trials,) movement task id
        "onsets": d["trial_onsets_samples"].astype(np.int64),  # (n_trials,) cue sample index
        "subject": os.path.basename(path).split("_")[0],
        "session": os.path.basename(path).split("_")[1].replace(".npz", ""),
    }


def epoch_trials(rec, cfg: Config):
    """Slice pre/post-cue windows for EEG and EMG from the same onset indices —
    shared clock guarantees alignment."""
    pre, post = int(cfg.epoch_pre_s * cfg.eeg_fs), int(cfg.epoch_post_s * cfg.eeg_fs)
    eeg_epochs, emg_epochs, labels = [], [], []
    for onset, label in zip(rec["onsets"], rec["labels"]):
        s, e = onset - pre, onset + post
        if s < 0 or e > len(rec["eeg"]):
            continue
        eeg_epochs.append(rec["eeg"][s:e])
        emg_epochs.append(rec["emg"][s:e])
        labels.append(label)
    return np.stack(eeg_epochs), np.stack(emg_epochs), np.array(labels)


## 4. Preprocessing (per modality)

EEG: bandpass 8–30 Hz + ICA against EOG (mandatory-control requirement). EMG: bandpass 20–450 Hz + notch.

In [ ]:
def sos_bandpass(fs, lo, hi, order=4):
    return butter(order, [lo, hi], btype="bandpass", fs=fs, output="sos")

def sos_notch(fs, freq, q=30.0):
    b, a = iirnotch(freq, q, fs)
    return tf2sos(b, a)


def preprocess_eeg(eeg_epochs, cfg: Config, eog_epochs=None):
    sos = sos_bandpass(cfg.eeg_fs, *cfg.eeg_bp)
    filtered = sosfiltfilt(sos, eeg_epochs, axis=1)
    if eog_epochs is not None:
        filtered = ica_denoise_epochs(filtered, eog_epochs, cfg)
    return filtered.astype(np.float32)


def ica_denoise_epochs(eeg_epochs, eog_epochs, cfg: Config):
    """Fit one ICA on the concatenated epochs, exclude components correlated with EOG.
    This is the mandatory EMG/EOG-contamination control for the EEG branch."""
    n_trials, n_time, n_ch = eeg_epochs.shape
    info = mne.create_info([f"EEG{i}" for i in range(n_ch)], cfg.eeg_fs, ch_types="eeg")
    concat = eeg_epochs.transpose(1, 0, 2).reshape(-1, n_ch).T  # (n_ch, n_trials*n_time)
    raw = mne.io.RawArray(concat, info, verbose=False)

    ica = mne.preprocessing.ICA(n_components=min(20, n_ch), random_state=RNG_SEED, max_iter="auto")
    ica.fit(raw)

    eog_concat = eog_epochs.transpose(1, 0, 2).reshape(-1, eog_epochs.shape[-1]).T
    eog_info = mne.create_info([f"EOG{i}" for i in range(eog_epochs.shape[-1])], cfg.eeg_fs, "eog")
    eog_raw = mne.io.RawArray(eog_concat, eog_info, verbose=False)
    raw.add_channels([eog_raw], force_update_info=True)

    eog_indices, _ = ica.find_bads_eog(raw, ch_name=eog_info["ch_names"])
    ica.exclude = eog_indices
    cleaned = ica.apply(raw.copy(), verbose=False)
    cleaned_data = cleaned.get_data(picks=[f"EEG{i}" for i in range(n_ch)])
    return cleaned_data.T.reshape(n_time, n_trials, n_ch).transpose(1, 0, 2)


def preprocess_emg(emg_epochs, cfg: Config):
    bp = sos_bandpass(cfg.emg_fs, *cfg.emg_bp)
    nt = sos_notch(cfg.emg_fs, cfg.emg_notch)
    out = sosfiltfilt(bp, emg_epochs, axis=1)
    out = sosfiltfilt(nt, out, axis=1)
    return out.astype(np.float32)


## 5. Feature extraction per branch

EEG: Riemannian covariance + tangent space (reused from Project 2). EMG: Hudgins time-domain set (reused from Project 1's feature functions — MAV/WL/ZC/SSC/RMS/WAMP).

In [ ]:
def eeg_riemann_features(eeg_epochs, fitted_cov=None, fitted_ts=None):
    """eeg_epochs: (N, T, C) -> transpose to (N, C, T) for pyriemann."""
    X = eeg_epochs.transpose(0, 2, 1)
    cov = fitted_cov or Covariances(estimator="oas").fit(X)
    covs = cov.transform(X)
    ts = fitted_ts or TangentSpace(metric="riemann").fit(covs)
    return ts.transform(covs), cov, ts


def _feat_mav(x):  return np.mean(np.abs(x), axis=0)
def _feat_wl(x):   return np.sum(np.abs(np.diff(x, axis=0)), axis=0)
def _feat_rms(x):  return np.sqrt(np.mean(x ** 2, axis=0))
def _feat_zc(x, th=1e-3):
    sc = (x[:-1] * x[1:]) < 0
    ok = np.abs(x[:-1] - x[1:]) > th
    return np.sum(sc & ok, axis=0)

def emg_hudgins_features(emg_epochs):
    """emg_epochs: (N, T, C) -> (N, C*4) flat feature matrix."""
    feats = []
    for ep in emg_epochs:
        feats.append(np.concatenate([_feat_mav(ep), _feat_wl(ep), _feat_rms(ep), _feat_zc(ep)]))
    return np.stack(feats)


## 6. Load + preprocess a subject, build EEG-only / EMG-only feature matrices

In [ ]:
files = list_subject_session_files()

if files:
    rec = load_session(files[0])
    eeg_ep, emg_ep, y = epoch_trials(rec, CFG)
    eeg_ep = preprocess_eeg(eeg_ep, CFG, eog_epochs=rec["eog"][None] if rec["eog"] is not None else None)
    emg_ep = preprocess_emg(emg_ep, CFG)

    from sklearn.model_selection import train_test_split
    idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.3,
                                            stratify=y, random_state=RNG_SEED)

    F_eeg, eeg_cov, eeg_ts = eeg_riemann_features(eeg_ep[idx_train])
    F_eeg_test, _, _ = eeg_riemann_features(eeg_ep[idx_test], fitted_cov=eeg_cov, fitted_ts=eeg_ts)

    F_emg = emg_hudgins_features(emg_ep[idx_train])
    F_emg_test = emg_hudgins_features(emg_ep[idx_test])

    y_train, y_test = y[idx_train], y[idx_test]
    print(f"EEG feature dim: {F_eeg.shape[1]}, EMG feature dim: {F_emg.shape[1]}")
else:
    print("Populate DATA_DIR with converted Jeong et al. sessions, then re-run.")


## 7. Three unimodal/fused models, three fusion levels

In [ ]:
def fit_eval_logreg(F_train, y_train, F_test, y_test):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(F_train, y_train)
    pred = clf.predict(F_test)
    proba = clf.predict_proba(F_test)
    return clf, pred, proba


def score(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro")}


class TwoBranchGatedNet(nn.Module):
    """Intermediate fusion: per-branch encoder, a learned scalar gate weighting
    the EEG branch vs. the EMG branch before the shared classification head."""
    def __init__(self, eeg_dim, emg_dim, n_classes, hidden=32):
        super().__init__()
        self.eeg_enc = nn.Sequential(nn.Linear(eeg_dim, hidden), nn.ReLU())
        self.emg_enc = nn.Sequential(nn.Linear(emg_dim, hidden), nn.ReLU())
        self.gate = nn.Sequential(nn.Linear(hidden * 2, 1), nn.Sigmoid())
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, eeg_x, emg_x):
        e = self.eeg_enc(eeg_x)
        m = self.emg_enc(emg_x)
        g = self.gate(torch.cat([e, m], dim=1))  # (B, 1) in [0, 1]
        fused = g * e + (1 - g) * m
        return self.head(fused), g.squeeze(-1)


def fit_gated_fusion(F_eeg_train, F_emg_train, y_train, n_classes, epochs=100, lr=1e-3):
    model = TwoBranchGatedNet(F_eeg_train.shape[1], F_emg_train.shape[1], n_classes)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    xe = torch.tensor(F_eeg_train, dtype=torch.float32)
    xm = torch.tensor(F_emg_train, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.long)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        out, _ = model(xe, xm)
        loss = loss_fn(out, yt)
        loss.backward()
        opt.step()
    return model


if files:
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(y_train)
    y_train_i, y_test_i = le.transform(y_train), le.transform(y_test)
    n_classes = len(le.classes_)

    eeg_scaler = StandardScaler().fit(F_eeg)
    emg_scaler = StandardScaler().fit(F_emg)
    F_eeg_s, F_eeg_test_s = eeg_scaler.transform(F_eeg), eeg_scaler.transform(F_eeg_test)
    F_emg_s, F_emg_test_s = emg_scaler.transform(F_emg), emg_scaler.transform(F_emg_test)

    fusion_results = []

    # unimodal
    _, pred, proba_eeg = fit_eval_logreg(F_eeg_s, y_train_i, F_eeg_test_s, y_test_i)
    fusion_results.append({"model": "eeg_only", **score(y_test_i, pred)})

    _, pred, proba_emg = fit_eval_logreg(F_emg_s, y_train_i, F_emg_test_s, y_test_i)
    fusion_results.append({"model": "emg_only", **score(y_test_i, pred)})

    # feature-concat fusion
    F_concat_train = np.concatenate([F_eeg_s, F_emg_s], axis=1)
    F_concat_test = np.concatenate([F_eeg_test_s, F_emg_test_s], axis=1)
    _, pred, _ = fit_eval_logreg(F_concat_train, y_train_i, F_concat_test, y_test_i)
    fusion_results.append({"model": "fusion_feature_concat", **score(y_test_i, pred)})

    # decision-level: probability averaging
    proba_avg = (proba_eeg + proba_emg) / 2
    pred_avg = proba_avg.argmax(axis=1)
    fusion_results.append({"model": "fusion_decision_avg", **score(y_test_i, pred_avg)})

    # decision-level: stacking (logreg on top of concatenated probabilities)
    from sklearn.model_selection import cross_val_predict
    stack_clf = LogisticRegression(max_iter=2000)
    stack_clf.fit(np.concatenate([proba_eeg, proba_emg], axis=1), y_test_i)  # meta-fit on held-out for demo
    # NOTE: for a rigorous stacking meta-learner, fit on cross-val OOF predictions from the
    # training fold instead — simplified here to a single split for clarity.

    # intermediate: two-branch gated network
    gated_model = fit_gated_fusion(F_eeg_s, F_emg_s, y_train_i, n_classes)
    gated_model.eval()
    with torch.no_grad():
        out, gate_vals = gated_model(torch.tensor(F_eeg_test_s, dtype=torch.float32),
                                      torch.tensor(F_emg_test_s, dtype=torch.float32))
        pred_gated = out.argmax(dim=1).numpy()
    fusion_results.append({"model": "fusion_intermediate_gated", **score(y_test_i, pred_gated)})
    print(f"mean learned gate weight on EEG branch: {gate_vals.mean().item():.3f} "
          f"(0 = fully EMG, 1 = fully EEG)")

    display(pd.DataFrame(fusion_results))


## 8. Systematic EMG degradation — the clinically meaningful result

Drop channels, inject noise, attenuate amplitude; plot accuracy vs. degradation for EMG-only, EEG-only, and fused. The crossover point is the finding.

In [ ]:
def degrade_drop_channels(emg, n_keep):
    idx = np.random.RandomState(RNG_SEED).choice(emg.shape[-1], size=n_keep, replace=False)
    out = np.zeros_like(emg)
    out[..., idx] = emg[..., idx]
    return out


def degrade_gaussian_noise(emg, snr_db):
    sig_power = np.mean(emg ** 2)
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise = np.random.RandomState(RNG_SEED).normal(0, np.sqrt(noise_power), emg.shape)
    return emg + noise


def degrade_attenuate(emg, factor):
    return emg * factor


def run_degradation_sweep(degrade_fn, param_values, param_name):
    rows = []
    for val in param_values:
        emg_test_degraded = degrade_fn(emg_ep[idx_test], val)
        F_emg_deg = emg_hudgins_features(emg_test_degraded)
        F_emg_deg_s = emg_scaler.transform(F_emg_deg)

        emg_clf = LogisticRegression(max_iter=2000).fit(F_emg_s, y_train_i)
        pred_emg = emg_clf.predict(F_emg_deg_s)
        proba_emg_deg = emg_clf.predict_proba(F_emg_deg_s)

        proba_fused = (proba_eeg + proba_emg_deg) / 2
        pred_fused = proba_fused.argmax(axis=1)

        rows.append({param_name: val,
                     "emg_only": accuracy_score(y_test_i, pred_emg),
                     "eeg_only": accuracy_score(y_test_i, proba_eeg.argmax(axis=1)),
                     "fusion_decision_avg": accuracy_score(y_test_i, pred_fused)})
    return pd.DataFrame(rows)


if files:
    channel_sweep = run_degradation_sweep(
        lambda emg, n: degrade_drop_channels(emg, n), range(CFG.emg_channels, 0, -1), "n_channels_kept")
    snr_sweep = run_degradation_sweep(
        lambda emg, snr: degrade_gaussian_noise(emg, snr), [20, 10, 5, 0, -5, -10], "snr_db")
    atten_sweep = run_degradation_sweep(
        lambda emg, f: degrade_attenuate(emg, f), [1.0, 0.5, 0.25, 0.1, 0.05, 0.0], "amplitude_factor")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, df, xcol, title in [
        (axes[0], channel_sweep, "n_channels_kept", "Channel dropout"),
        (axes[1], snr_sweep, "snr_db", "Gaussian noise (SNR)"),
        (axes[2], atten_sweep, "amplitude_factor", "Amplitude attenuation"),
    ]:
        for col in ["emg_only", "eeg_only", "fusion_decision_avg"]:
            ax.plot(df[xcol], df[col], "o-", label=col)
        ax.set_xlabel(xcol)
        ax.set_ylabel("accuracy")
        ax.set_title(title)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig("project3_degradation_crossover.png", dpi=150)
    plt.show()

    for df, xcol in [(channel_sweep, "n_channels_kept"), (snr_sweep, "snr_db"), (atten_sweep, "amplitude_factor")]:
        crossover = df[df["fusion_decision_avg"] > df["emg_only"]]
        if len(crossover):
            print(f"Fusion overtakes EMG-alone at {xcol} <= {crossover[xcol].iloc[0]}")
        else:
            print(f"No crossover found across the swept {xcol} range — fusion never beats EMG-alone here.")


## 9. Timing asymmetry: accuracy vs. time relative to cue, per modality

Movement-related cortical activity precedes muscle activation by a few hundred ms. If EEG carries usable intent before EMG onset, that supports EEG-as-early-trigger + EMG-as-confirmation.

In [ ]:
def sliding_prewindow_accuracy(eeg_full, emg_full, y, onsets_rel_to_epoch_start, cfg: Config,
                                window_s=0.3, step_s=0.1, t_range=(-0.8, 0.8)):
    """Slide a window_s window across the epoch, re-extract features and re-fit/evaluate
    at each time offset relative to cue, for EEG and EMG independently."""
    fs = cfg.eeg_fs
    win = int(window_s * fs)
    offsets = np.arange(t_range[0], t_range[1], step_s)
    rows = []
    for off in offsets:
        center = onsets_rel_to_epoch_start + int(off * fs)
        s, e = center - win // 2, center + win // 2
        if s < 0 or e > eeg_full.shape[1]:
            continue
        eeg_win = eeg_full[:, s:e, :]
        emg_win = emg_full[:, s:e, :]

        F_eeg_t, _, _ = eeg_riemann_features(eeg_win)
        F_emg_t = emg_hudgins_features(emg_win)

        from sklearn.model_selection import cross_val_score
        eeg_acc = cross_val_score(LogisticRegression(max_iter=1000), F_eeg_t, y, cv=3).mean()
        emg_acc = cross_val_score(LogisticRegression(max_iter=1000), F_emg_t, y, cv=3).mean()
        rows.append({"t_rel_cue_s": off, "eeg_accuracy": eeg_acc, "emg_accuracy": emg_acc})
    return pd.DataFrame(rows)


if files:
    cue_sample = int(CFG.epoch_pre_s * CFG.eeg_fs)  # cue is at epoch_pre_s into each epoch
    timing_df = sliding_prewindow_accuracy(eeg_ep, emg_ep, y, cue_sample, CFG)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(timing_df["t_rel_cue_s"], timing_df["eeg_accuracy"], "o-", label="EEG")
    ax.plot(timing_df["t_rel_cue_s"], timing_df["emg_accuracy"], "o-", label="EMG")
    ax.axvline(0, color="gray", linestyle="--", label="cue")
    ax.set_xlabel("time relative to cue (s)")
    ax.set_ylabel("classification accuracy")
    ax.set_title("Pre-movement decodability: EEG vs. EMG")
    ax.legend()
    plt.tight_layout()
    plt.savefig("project3_timing_asymmetry.png", dpi=150)
    plt.show()

    pre_cue = timing_df[timing_df["t_rel_cue_s"] < 0]
    if len(pre_cue):
        eeg_lead = pre_cue[pre_cue["eeg_accuracy"] > pre_cue["emg_accuracy"] + 0.05]
        if len(eeg_lead):
            print(f"EEG decodes above chance+margin before EMG at t = {eeg_lead['t_rel_cue_s'].iloc[0]:.2f}s "
                  f"relative to cue — candidate early-trigger window.")


## 10. Mandatory control: EEG branch tested on pre-movement-only windows

Confirms the EEG classifier isn't secretly reading EMG bleed-through / muscle artifact — evaluate it restricted to windows strictly before any EMG onset.

In [ ]:
def emg_onset_sample(emg_trial, baseline_samples, k_std=3.0):
    """First sample where rectified EMG exceeds baseline mean + k*std, per trial."""
    baseline = np.abs(emg_trial[:baseline_samples]).mean(axis=0)
    baseline_std = np.abs(emg_trial[:baseline_samples]).std(axis=0)
    thresh = baseline + k_std * baseline_std
    rectified = np.abs(emg_trial)
    exceeds = (rectified > thresh).any(axis=1)
    onset_idx = np.argmax(exceeds) if exceeds.any() else len(emg_trial)
    return onset_idx


if files:
    baseline_n = int(0.3 * CFG.eeg_fs)  # first 300ms of the epoch as EMG-quiet baseline
    onsets = np.array([emg_onset_sample(trial, baseline_n) for trial in emg_ep])

    pre_movement_eeg = []
    for i, onset in enumerate(onsets):
        cutoff = min(onset, cue_sample) if onset > baseline_n else cue_sample
        pre_movement_eeg.append(eeg_ep[i, :cutoff, :].mean(axis=0))  # crude fixed-dim summary
    # NOTE: for a rigorous version, re-run eeg_riemann_features on a common fixed pre-onset
    # window length (e.g. the shortest onset across trials) rather than this mean-summary stand-in.

    pre_movement_eeg = np.stack(pre_movement_eeg)
    from sklearn.model_selection import cross_val_score
    control_acc = cross_val_score(LogisticRegression(max_iter=1000), pre_movement_eeg, y, cv=3).mean()
    print(f"EEG accuracy restricted to pre-EMG-onset windows: {control_acc:.1%} "
          f"(chance = {1/CFG.n_classes:.1%}). Above-chance here supports a genuine EEG "
          f"movement-intent signal, not EMG contamination of the EEG channels.")


## Notes / expected findings

- If **feature-concat fusion barely beats EMG-only**, that's the expected result, not a failure — EMG's SNR advantage dominates naive fusion. The degradation sweep (section 8) is where the real story lives: report the **crossover point**, not the headline accuracy.
- The gated intermediate-fusion model's learned gate weight (section 7) is itself a diagnostic — it should shift toward EEG as EMG degrades, if you re-run fusion training per degradation level rather than reusing weights fit on clean EMG.
- Section 10's control is a placeholder implementation (per-trial mean-summary rather than a proper fixed-length re-windowing) — tighten it before treating the pre-movement-only accuracy number as a paper-ready result.
- Swap in the WAY-EEG-GAL backup dataset by replacing `load_session`/`epoch_trials` with loaders for its distributed format if GigaDB access to Jeong et al. is unavailable.
